In [1]:
# ==============================================================================
# BLOCO UNIFICADO DE IMPORTAÇÕES - ETAPA DE PREPARAÇÃO DE IMAGENS
# ==============================================================================

# 1. BIBLIOTECAS PADRÃO (STANDARD LIBRARY)
import json
import os
import random
import time

# 2. MANIPULAÇÃO DE DADOS E REQUISIÇÕES
import numpy as np
import pandas as pd
import requests

# 3. GEOPROCESSAMENTO E SENSORIAMENTO REMOTO
import ee
import geopandas as gpd
import osmnx as ox
import rasterio
from shapely.geometry import box

# 4. PROCESSAMENTO DE IMAGEM
from PIL import Image

In [1]:
def process_full(img):
    img = ee.Image(img)
    qa = img.select('QA60')
    cloud = qa.bitwiseAnd(1 << 10).eq(0)
    cirrus = qa.bitwiseAnd(1 << 11).eq(0)
    qa_mask = cloud.And(cirrus)

    scl = img.select('SCL')
    scl_mask = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))

    mask = qa_mask.And(scl_mask)

    # 14 CANAIS COMPLETO (3 TCI + 11 BUSTO/SPECTRAL INCLUINDO B8):
    return img.updateMask(mask).select([
        'TCI_R',
        'TCI_G',
        'TCI_B',
        'B1',
        'B2',
        'B3',
        'B4',
        'B5',
        'B6',
        'B7',
        'B8',
        'B8A',
        'B11',
        'B12',
    ]).toUint16()


In [ ]:
# =========================================================
# IMPORTS
# =========================================================

import os
import random
import time
import ee
import geopandas as gpd
import numpy as np
import osmnx as ox
import pandas as pd
import requests
from shapely.geometry import box

# =========================================================
# INIT & CONFIG
# =========================================================

ox.settings.requests_timeout = 60
ox.settings.max_query_area_size = 10**12
ox.settings.use_cache = True
ox.settings.log_console = False

try:
    ee.Initialize(project='desafio-solved')
except:
    ee.Authenticate()
    ee.Initialize(project='desafio-solved')

PATCH_SIZE_METERS = 2560
RESOLUTION = 10
proj_metrica = ee.Projection('EPSG:3857')

# Diretorios no Google Drive
DRIVE_FOLDER_images = 'Dataset_Pistas_Landing_2/images'
DRIVE_FOLDER_masks = 'Dataset_Pistas_Landing_2/masks'

# =========================================================
# CARREGAMENTO DO ASSET (SUAS 2561 COORDENADAS ORIGINAIS)
# =========================================================

pistas_asset = ee.FeatureCollection(
    'projects/desafio-solved/assets/Pistas_de_Pouso'
)
municipios = ee.FeatureCollection('FAO/GAUL/2015/level2')

excluir = ['Itaituba', 'Jacareacanga']
muni_excluir = municipios.filter(ee.Filter.inList('ADM2_NAME', excluir))
pistas_validas_ee = pistas_asset.filter(ee.Filter.bounds(muni_excluir).Not())

print('Obtendo as coordenadas originais do Asset...')
info_pistas = pistas_validas_ee.geometry().coordinates().getInfo()
print(f'Total de coordenadas obtidas: {len(info_pistas)}')

# Converter para GeoDataFrame local
lons = [coord[0] for coord in info_pistas]
lats = [coord[1] for coord in info_pistas]

pts_originais = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(lons, lats), crs='EPSG:4326'
)

# =========================================================
# CRIAÇÃO DE BUFFERS OTIMIZADOS E AGREGADOS
# =========================================================

print('Calculando buffers ao redor das coordenadas reais')
pts_projetados = pts_originais.to_crs(epsg=3857)

# criar fusões regionais consistentes e reduzir requisições
buffers_metricos = pts_projetados.geometry.buffer(1500)
buffers_gdf = gpd.GeoDataFrame(
    geometry=buffers_metricos, crs='EPSG:3857'
).to_crs(epsg=4326)

print('Dissolvendo polígonos sobrepostos para otimizar as requisições')
buffers_dissolvidos = buffers_gdf.dissolve()
blocos_ativos = buffers_dissolvidos.explode(index_parts=False).reset_index(
    drop=True
)

total_blocos = len(blocos_ativos)
print(
    f'As {len(info_pistas)} coordenadas originais foram consolidadas em {total_blocos} áreas de busca focadas.'
)

# =========================================================
# BUSCA DIRECIONADA NO OSM COM CACHE E DETECÇÃO DE ÁREAS VAZIAS
# =========================================================

try:
    major_version = int(ox.__version__.split('.')[0])
except Exception:
    major_version = 1

CACHE_FILE = 'cache_osm_pistas.gpkg'
runways_local_list = []
blocos_processados = set()

if os.path.exists(CACHE_FILE):
    print(
        f"Encontrado progresso anterior em '{CACHE_FILE}'. Carregando dados..."
    )
    try:
        df_cache = gpd.read_file(CACHE_FILE)
        if not df_cache.empty:
            blocos_processados = set(
                df_cache['bloco_origem_idx'].unique().astype(int)
            )
            runways_local_list.append(df_cache)
            print(
                f'-> {len(blocos_processados)} blocos já haviam sido processados com sucesso.'
            )
    except Exception as e:
        print(f'Erro ao ler arquivo de cache: {e}. Recomeçando...')

print('Iniciando busca direcionada no OSM...')

for idx, row in blocos_ativos.iterrows():
    if idx in blocos_processados:
        continue

    # Limpeza preventiva de geometria
    cell_geom = row.geometry.buffer(0)
    west, south, east, north = cell_geom.bounds

    max_tentativas = 5
    tentativa = 0
    sucesso = False
    tempo_espera = 3.0
    fator_reducao = 1.0

    while tentativa < max_tentativas and not sucesso:
        try:
            if tentativa >= 2 and fator_reducao == 1.0:
                print(
                    f'-> [Otimização] Reduzindo área de busca do bloco {idx+1} para aliviar servidor...'
                )
                fator_reducao = 0.5
                centroid = cell_geom.centroid
                largura = (east - west) * fator_reducao
                altura = (north - south) * fator_reducao
                west = centroid.x - (largura / 2)
                east = centroid.x + (largura / 2)
                south = centroid.y - (altura / 2)
                north = centroid.y + (altura / 2)

            if major_version >= 2:
                bbox_param = (west, south, east, north)
            else:
                bbox_param = (north, south, east, west)

            ox.settings.requests_timeout = 90 if tentativa == 0 else 180
            runways_bloco = ox.features_from_bbox(
                bbox=bbox_param, tags={'aeroway': 'runway'}
            )

            # Se encontrou pistas
            if not runways_bloco.empty:
                runways_bloco = runways_bloco[['geometry']].copy()
                runways_bloco['bloco_origem_idx'] = idx
                runways_local_list.append(runways_bloco)

                if not os.path.exists(CACHE_FILE):
                    runways_bloco.to_file(CACHE_FILE, driver='GPKG')
                else:
                    runways_bloco.to_file(CACHE_FILE, driver='GPKG', mode='a')

            sucesso = True
            blocos_processados.add(idx)
            print(
                f'Progresso OSM: {idx+1}/{total_blocos} processado. (Pistas encontradas!)'
            )

        except Exception as e:
            erro_msg = str(e).lower()

            # --- CASO 1: ÁREA VAZIA (Não é falha física!) ---
            if (
                'no matching features' in erro_msg
                or 'found no features' in erro_msg
            ):
                from shapely.geometry import LineString

                dummy_linha = LineString(
                    [(west, south), (west + 0.00001, south + 0.00001)]
                )
                dummy = gpd.GeoDataFrame(
                    {'bloco_origem_idx': [idx]},
                    geometry=[dummy_linha],
                    crs='EPSG:4326',
                )

                if not os.path.exists(CACHE_FILE):
                    dummy.to_file(CACHE_FILE, driver='GPKG')
                else:
                    dummy.to_file(CACHE_FILE, driver='GPKG', mode='a')

                sucesso = True
                blocos_processados.add(idx)
                print(
                    f'Progresso OSM: {idx+1}/{total_blocos} processado. (Área vazia no OSM - salva com sucesso)'
                )
                break  # Sai do loop de retentativas imediatamente

            # --- CASO 2: ERROS DE CONEXÃO OU LIMITE DE REQUISIÇÕES ---
            tentativa += 1
            if '429' in erro_msg:
                tempo_espera = max(tempo_espera, 35.0)
                print(
                    f'\n[Bloqueio 429] Muitas requisições. Resfriando por {tempo_espera}s...'
                )

            if tentativa < max_tentativas:
                print(
                    f'[Aviso] Falha de conexão na área {idx+1}/{total_blocos} (Tentativa {tentativa}/{max_tentativas}).'
                )
                time.sleep(tempo_espera)
                tempo_espera *= 2.0
            else:
                print(
                    f'\n[AVISO CRÍTICO] Área {idx+1} instável. Pulando e salvando como vazia para prosseguir...'
                )
                from shapely.geometry import LineString

                dummy_linha = LineString(
                    [(west, south), (west + 0.00001, south + 0.00001)]
                )
                dummy = gpd.GeoDataFrame(
                    {'bloco_origem_idx': [idx]},
                    geometry=[dummy_linha],
                    crs='EPSG:4326',
                )
                if not os.path.exists(CACHE_FILE):
                    dummy.to_file(CACHE_FILE, driver='GPKG')
                else:
                    dummy.to_file(CACHE_FILE, driver='GPKG', mode='a')
                blocos_processados.add(idx)

    time.sleep(random.uniform(0.5, 2.0))

# Carregar tudo o que foi acumulado no cache local do Geopackage
if os.path.exists(CACHE_FILE):
    runways_all = gpd.read_file(CACHE_FILE)
    runways_all = runways_all[
        runways_all.geometry.type.isin(
            ['Polygon', 'MultiPolygon', 'LineString', 'MultiLineString']
        )
    ]
else:
    print('Nenhuma pista foi encontrada ou salva em cache.')
    exit()

runways_all['geom_wkt'] = runways_all.geometry.to_wkt()
runways_all = (
    runways_all.drop_duplicates(subset=['geom_wkt'])
    .drop(columns=['geom_wkt', 'bloco_origem_idx'], errors='ignore')
    .reset_index(drop=True)
)

print(
    f'Busca finalizada! Total de geometrias únicas coletadas: {len(runways_all)}'
)

# =========================================================
# FILTRAGEM ESPACIAL E TRATAMENTO DE GEOMETRIAS
# =========================================================

# 2. Buffer dos pontos diretamente em EPSG:3857 (metros)
pts_buffer = pts_originais.to_crs(epsg=3857)
pts_buffer['geometry'] = pts_buffer.geometry.buffer(1280)

# 3. Converte pistas para metros antes da junção espacial
runways_temp = runways_all.to_crs(epsg=3857)

# 4. Junção espacial em metros
runways_intersect = gpd.sjoin(
    runways_temp, pts_buffer, how='inner', predicate='intersects'
)

if runways_intersect.empty:
    print('Nenhuma pista restou após a filtragem espacial local.')
    exit()

runways = runways_intersect.copy()

# 5. Aplicação do Buffer
novas_geometrias = []
for geom in runways.geometry:
    if geom.geom_type in ['LineString', 'MultiLineString']:
        novas_geometrias.append(geom.buffer(20))
    else:
        novas_geometrias.append(geom.buffer(5))

runways_temp = runways.copy()
runways_temp['geometry'] = novas_geometrias

# 6. Dissolve e unificação de geometrias
runways_limpas = runways_temp.dissolve()
runways_unicas = runways_limpas.explode(index_parts=False).reset_index(
    drop=True
)

# 7. Filtro por área mínima de 5000 m²
runways_unicas['area'] = runways_unicas.geometry.area
runways_filtradas = runways_unicas[runways_unicas['area'] > 5000].copy()

if runways_filtradas.empty:
    print('Nenhuma pista restou após o filtro de área mínima de 5000 m².')
    exit()

# 8. Reconciliação final para WGS84
runways = runways_filtradas.to_crs(epsg=4326)

features = []
for _, row in runways.iterrows():
    geom = row.geometry
    geojson = geom.__geo_interface__
    ee_geom = ee.Geometry(geojson)
    features.append(ee.Feature(ee_geom))

pistas = ee.FeatureCollection(features)

# =========================================================
# PROCESSAMENTO SENTINEL-2 (COMPATIBILIDADE SATLAS + B8)
# =========================================================

limites_locais = runways.total_bounds
ee_bounds = ee.Geometry.BBox(
    limites_locais[0], limites_locais[1], limites_locais[2], limites_locais[3]
)

s2Col = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterDate('2022-01-01', '2024-12-31')
    .filter(
        ee.Filter.calendarRange(7, 11, 'month')
    )
    .filterBounds(ee_bounds)
    .map(process_full)
)

# =========================================================
# FUNÇÃO DE EXPORTAÇÃO
# =========================================================

def export_to_google_drive(
    img,
    mask,
    region,
    suffix,
    folder_img,
    folder_mask,
    native_crs,
    native_transform,
):
    """Exporta usando o CRS nativo e o crsTransform da imagem.

    Alinha os pixels perfeitamente na grade UTM do satélite.
    """
    task_img = ee.batch.Export.image.toDrive(
        image=img.clip(region),
        description=f'positivo_imagem_{suffix}',
        folder=folder_img,
        fileNamePrefix=f'positivo_imagem_{suffix}',
        crs=native_crs,
        crsTransform=native_transform,
        region=region,
        maxPixels=1e13,
    )
    task_img.start()

    task_mask = ee.batch.Export.image.toDrive(
        image=mask.clip(region),
        description=f'positivo_mascara_{suffix}',
        folder=folder_mask,
        fileNamePrefix=f'positivo_mascara_{suffix}',
        crs=native_crs,
        crsTransform=native_transform,
        region=region,
        maxPixels=1e13,
    )
    task_mask.start()


# =========================================================
# LOOP DE EXPORTAÇÃO
# =========================================================

pistas_list = pistas.toList(pistas.size())
num_pistas = pistas.size().getInfo()

print(f'\nTotal de pistas tratadas encontradas: {num_pistas}')
print(
    f'Iniciando agendamento individual de tarefas no Google Drive usando CRS nativo...'
)

for i in range(num_pistas):
    try:
        feature = ee.Feature(pistas_list.get(i))
        geom = feature.geometry()
        centro = geom.centroid()

        regiao_filtro = centro.buffer(PATCH_SIZE_METERS).bounds()

        # Busca de cenas limpas na coleção restrita da estação seca
        colecao_local = s2Col.filterBounds(regiao_filtro).filter(
            ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)
        )

        # 1. OBTER EXTRAÇÃO DA PROJEÇÃO
        imagem_referencia = ee.Image(
            ee.Algorithms.If(
                colecao_local.size().gt(0),
                colecao_local.sort('CLOUDY_PIXEL_PERCENTAGE').first(),
                s2Col.filterBounds(regiao_filtro)
                .sort('CLOUDY_PIXEL_PERCENTAGE')
                .first(),
            )
        )

        # Seleciona TCI_R para obter a projeção UTM nativa
        proj_info = imagem_referencia.select('TCI_R').projection().getInfo()
        crs_nativo = proj_info['crs']
        transform_nativo = proj_info['transform']

        proj_nativa = ee.Projection(crs_nativo, transform_nativo)

        # 2. CONTEÚDO DA IMAGEM
        imagem_local = ee.Image(
            ee.Algorithms.If(
                colecao_local.size().gt(0),
                colecao_local.sort('CLOUDY_PIXEL_PERCENTAGE').first(),
                s2Col.filterBounds(regiao_filtro).median(),
            )
        )

        # 3. MASCARAMENTO E GEOMETRIA DO PATCH
        dx = random.randint(-50, 50)
        dy = random.randint(-50, 50)

        coords = centro.transform(proj_nativa, 1).coordinates()
        novo_ponto = ee.Geometry.Point(
            [
                ee.Number(coords.get(0)).add(dx),
                ee.Number(coords.get(1)).add(dy),
            ],
            proj_nativa,
        )

        regiao = novo_ponto.buffer(PATCH_SIZE_METERS / 2).bounds(1, proj_nativa)

        # 4. MÁSCARA DA PISTA
        mask = ee.Image.constant(0).paint(feature, 1).rename('label').uint8()

        # 5. AGENDAMENTO NO GOOGLE DRIVE
        suffix_str = f'{i:05d}'
        export_to_google_drive(
            imagem_local,
            mask,
            regiao,
            suffix_str,
            DRIVE_FOLDER_images,
            DRIVE_FOLDER_masks,
            crs_nativo,
            transform_nativo,
        )

        print(
            f'[{i+1}/{num_pistas}] Tarefa agendada com sucesso: "positivo_imagem_{suffix_str}" em CRS: {crs_nativo}'
        )

    except Exception as e:
        print(f'ERRO Crítico na pista {i}: {e}')

In [ ]:
import json
import ee

# =====================================================================
# 1. INICIALIZAÇÃO E CONFIGURAÇÕES DE PASTA
# =====================================================================
PROJECT_ID = 'desafio-solved'

try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

GEOJSON_PATH = 'falsos_positivos_Brasil_Novo.geojson'

DRIVE_FOLDER_IMAGENS = 'GEE_Dataset_Negativos_Imagens'
DRIVE_FOLDER_MASCARAS = 'GEE_Dataset_Negativos_Mascaras'

# =====================================================================
# 3. CARREGAMENTO DO GEOJSON DE FALSOS POSITIVOS
# =====================================================================
print(f'Lendo arquivo de coordenadas: {GEOJSON_PATH}...')

with open(GEOJSON_PATH, 'r') as f:
    geojson_data = json.load(f)

features = geojson_data.get('features', [])
total_pontos = len(features)
print(f'Total de pontos de falsos positivos encontrados: {total_pontos}')

if total_pontos == 0:
    raise ValueError(
        'Nenhum ponto encontrado no GeoJSON. Verifique o arquivo.'
    )


# =====================================================================
# 4. PREPARAÇÃO DO COMPOSIÇÃO MEDIANA (BRASIL NOVO)
# =====================================================================
municipios_asset = ee.FeatureCollection('FAO/GAUL/2015/level2')
roi_brasil_novo = municipios_asset.filter(
    ee.Filter.eq('ADM2_NAME', 'Brasil Novo')
).geometry()

# Coleção de imagens do período seco
s2Col = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(roi_brasil_novo)
    .filterDate('2022-01-01', '2024-12-31')
    .filter(ee.Filter.calendarRange(7, 11, 'month'))
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 25))
    .map(process_full)
)

# Imagem mediana de 14 canais
imagem_mediana = s2Col.median()

# Projeção nativa UTM para manter integridade espacial
primeira_imagem = s2Col.first()
proj_info = primeira_imagem.select('TCI_R').projection().getInfo()
crs_nativo = proj_info['crs']

# Imagem de Máscara Totalmente Preta (0s) de 1 canal
mascara_preta = ee.Image.constant(0).toByte().rename('mask')


# =====================================================================
# 5. LOOP DE GERACÃO E EXPORTACÃO DOS PATCHES 256x256
# =====================================================================
# Tamanho do patch em metros: 256 pixels * 10m/pixel = 2560m
RAIO_METROS = 1280  # Metade da largura/altura do patch (2560 / 2)

print('\nAgendando tarefas de exportação para cada falso positivo...')

for idx, feat in enumerate(features, start=1):
    coords = feat['geometry']['coordinates']
    lon, lat = coords[0], coords[1]

    # Ponto central e Bounding Box quadrado de 2560m x 2560m (256x256 px a 10m)
    ponto_ee = ee.Geometry.Point([lon, lat])
    patch_bbox = ponto_ee.buffer(RAIO_METROS).bounds()

    # Formatação do nome com 5 dígitos (ex: 00001, 00063)
    str_index = f'{idx:05d}'
    nome_imagem = f'negativo_imagem_{str_index}'
    nome_mascara = f'negativo_mascara_{str_index}'

    # -----------------------------------------------------------------
    # Exportação 1: Imagem de Inferência (14 canais, 256x256)
    # -----------------------------------------------------------------
    task_img = ee.batch.Export.image.toDrive(
        image=imagem_mediana,
        description=f'Export_{nome_imagem}',
        folder=DRIVE_FOLDER_IMAGENS,
        fileNamePrefix=nome_imagem,
        region=patch_bbox,
        dimensions='256x256',
        crs=crs_nativo,
        maxPixels=1e9,
        fileFormat='GeoTIFF',
        formatOptions={'cloudOptimized': True},
    )
    task_img.start()

    # -----------------------------------------------------------------
    # Exportação 2: Máscara ZERADA (1 canal, uint8, 256x256)
    # -----------------------------------------------------------------
    task_mask = ee.batch.Export.image.toDrive(
        image=mascara_preta,
        description=f'Export_{nome_mascara}',
        folder=DRIVE_FOLDER_MASCARAS,
        fileNamePrefix=nome_mascara,
        region=patch_bbox,
        dimensions='256x256',
        crs=crs_nativo,
        maxPixels=1e9,
        fileFormat='GeoTIFF',
        formatOptions={'cloudOptimized': True},
    )
    task_mask.start()

    print(
        f'-> [{idx}/{total_pontos}] Agendados: {nome_imagem}.tif e {nome_mascara}.tif'
    )

print(
    '\n[CONCLUÍDO] Todas as tarefas de imagens e máscaras foram enviadas ao GEE!'
)

In [ ]:
import ee

# =====================================================================
# 1. INICIALIZAÇÃO E CONFIGURAÇÃO
# =====================================================================

PROJECT_ID = 'desafio-solved'

try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

# Lista de municípios para exportação
lista_municipios = [
    'Brasil Novo',
    'Jacareacanga',
    'Itaituba',
]

DRIVE_FOLDER = 'GEE_Desafio_Solved_Inference'

# =====================================================================
# 3. LOOP DE EXPORTAÇÃO POR MUNICÍPIO
# =====================================================================
municipios_asset = ee.FeatureCollection('FAO/GAUL/2015/level2')

for municipio_nome in lista_municipios:
    print(f'\n--- Processando município: {municipio_nome} ---')

    # Geometria do município (ROI)
    roi_fc = municipios_asset.filter(
        ee.Filter.eq('ADM2_NAME', municipio_nome)
    )

    if roi_fc.size().getInfo() == 0:
        print(
            f"[ERRO] Município '{municipio_nome}' não foi encontrado no dataset FAO/GAUL."
        )
        continue

    roi = roi_fc.geometry()

    # Coleção de imagens Sentinel-2 idêntica ao treino:
    s2Col = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(roi)
        .filterDate('2022-01-01', '2024-12-31')
        .filter(ee.Filter.calendarRange(7, 11, 'month'))
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 25))
        .map(process_full)
    )

    total_imgs = s2Col.size().getInfo()
    print(f'Total de imagens encontradas no período seco: {total_imgs}')

    if total_imgs == 0:
        print(
            f'[AVISO] Nenhuma imagem encontrada para {municipio_nome}. Pulando...'
        )
        continue

    # Captura da projeção nativa UTM a partir da primeira imagem da ROI (usando TCI_R como no treino)
    primeira_imagem = s2Col.first()
    proj_info = primeira_imagem.select('TCI_R').projection().getInfo()
    crs_nativo = proj_info['crs']
    transform_nativo = proj_info['transform']

    print(f'CRS Nativo identificado: {crs_nativo}')

    # Mediana do período seco (Garante composição contínua sem nuvens)
    imagem_mediana = s2Col.median()

    # Recorte na geometria do município
    imagem_final = imagem_mediana.clip(roi)

    # Configuração da Exportação
    nome_arquivo = (
        municipio_nome.replace(' ', '_').replace('-', '_').lower()
    )

    task = ee.batch.Export.image.toDrive(
        image=imagem_final,
        description=f'Export_{nome_arquivo}',
        folder=DRIVE_FOLDER,
        fileNamePrefix=f'inferencia_imagem_{nome_arquivo}',
        region=roi,
        crs=crs_nativo,
        crsTransform=transform_nativo,  # Garante o alinhamento de grade de 10m
        maxPixels=1e13,
        fileFormat='GeoTIFF',
        formatOptions={'cloudOptimized': True},
    )

    task.start()
    print(
        f'Tarefa agendada com sucesso para {municipio_nome} (Pasta: {DRIVE_FOLDER})'
    )

print(
    '\n[CONCLUÍDO] Todas as tarefas de exportação para inferência foram agendadas no GEE!'
)

In [ ]:
import ee
ee.Initialize(project='desafio-solved-v3')

# Obtém a lista de todas as tarefas
tasks = ee.batch.Task.list()

print(f"Verificando {len(tasks)} tarefas...")

canceladas = 0
for task in tasks:
    # Verifica se a tarefa está na fila (READY) ou rodando (RUNNING)
    if task.state == 'READY' or task.state == 'RUNNING':
        task.cancel()
        canceladas += 1

print(f"Sucesso! {canceladas} tarefas foram canceladas.")

In [ ]:
import numpy as np
import rasterio
from PIL import Image
import os

#'TCI_R',
#'TCI_G',
#'TCI_B',
#'B1',
#'B2',
#'B3',
#'B4',
#'B5',
#'B6',
#'B7',
#'B8',
#'B8A',
#'B11',
#'B12',


def normalizar_canal(banda):
    """Ajusta o contraste e converte a banda para o formato de 8 bits (0-255)

    necessário para o PNG.
    """
    # Remove valores inválidos (NaN/Inf) se existirem
    banda_valida = banda[np.isfinite(banda)]

    # Corta os 2% mais escuros e 2% mais claros para melhorar muito o contraste
    p_min = np.percentile(banda_valida, 2)
    p_max = np.percentile(banda_valida, 98)

    # Aplica o corte e redimensiona para 0-255
    banda_norm = np.clip((banda - p_min) / (p_max - p_min) * 255, 0, 255)
    return banda_norm.astype(np.uint8)

def salvar_imagem_rgb(caminho_tiff, caminho_saida):
    
    with rasterio.open(caminho_tiff) as src:
        azul = src.read(3)  # B2 é a primeira
        verde = src.read(2)  # B3 é a segunda
        vermelho = src.read(1)  # B4 é a terceira
    
        r_norm = normalizar_canal(vermelho)
        g_norm = normalizar_canal(verde)
        b_norm = normalizar_canal(azul)
    
        # Junta os três canais na ordem RGB (Red, Green, Blue)
        imagem_rgb = np.dstack((r_norm, g_norm, b_norm))
    
        # Salva o resultado final como PNG
        caminho_png = caminho_saida.replace('.tif', '.png')
        Image.fromarray(imagem_rgb).save(caminho_png)
    
        print(f"Sucesso! Imagem colorida normal salva como: {caminho_png}")

def salvar_mascara(caminho_tiff, caminho_saida):
    with rasterio.open(caminho_tiff) as src:
        # Como é uma máscara, lemos apenas a primeira banda (canal único)
        mascara = src.read(1)

        # Remove valores inválidos (NaN), substituindo-os por 0 (preto)
        mascara = np.nan_to_num(mascara, nan=0.0)

        # Se a sua máscara veio com valores 0 e 1, multiplicamos por 255
        # para que o que é 1 vire branco puro (255) no PNG
        if mascara.max() <= 1.0:
            mascara_visualivel = (mascara * 255).astype(np.uint8)
        else:
            # Caso ela já tenha valores maiores (ex: de 0 a 255 ou IDs de classes),
            # apenas garantimos que está no formato de imagem correto
            mascara_visualivel = mascara.astype(np.uint8)

        # Salva o resultado como um PNG em escala de cinza (Preto e Branco)
        caminho_png = caminho_saida.replace('.tif', '.png')
        Image.fromarray(mascara_visualivel).save(caminho_png)

        print(f"Máscara convertida com sucesso e salva como: {caminho_png}")

caminho_images = 'dataset/images'
caminho_masks = 'dataset/masks'
caminho_destino_positivo = 'dataset_png/images_png'
caminho_destino_mascara = 'dataset_png/mascaras_png'

quantidade = len(os.listdir(caminho_images))

for i in range(quantidade):
    caminho_positivo_tiff = os.path.join(caminho_images, f'positivo_imagem_{i+0000:05d}.tif')
    caminho_positivo_saida = os.path.join(caminho_destino_positivo, f'PNG_positivo_{i+0000:05d}.png')
    caminho_mascara_tiff = os.path.join(caminho_masks , f'positivo_mascara_{i+0000:05d}.tif')
    caminho_mascara_saida = os.path.join(caminho_destino_mascara, f'PNG_mascara_{i+0000:05d}.png')

    if os.path.exists(caminho_positivo_tiff):
        salvar_imagem_rgb(caminho_positivo_tiff, caminho_positivo_saida)
        salvar_mascara(caminho_mascara_tiff, caminho_mascara_saida)
    else:
        print(f"Aviso: Arquivo {caminho_positivo_tiff} não encontrado. Pulando...")
        print(f"Aviso: Arquivo {caminho_mascara_tiff} não encontrado. Pulando...")

